# 04 — Baselines e Classificadores Clássicos

Implementa a Fase 18 do plano de elaboração: treina os cinco
classificadores clássicos configurados
(`configs/model_params.yaml -> classical`) sobre a matriz TF-IDF e avalia
cada um rigorosamente (matriz de confusão, curvas ROC, IC 95% via
bootstrap — ver `CLAUDE.md` -> "Rigorous evaluation").

**Sobre o conjunto de avaliação**: o projeto ainda não persiste uma
transformação TF-IDF reutilizável para texto fora do vocabulário de
treino (ver docstring de `src/main.py`), então `paths.test_corpus_file`
não pode ser vetorizado aqui. Este notebook usa, em vez disso, uma
partição interna de holdout sobre o próprio conjunto de treino
(`data.splitter.create_stratified_split`), estratificada e com semente
fixa — mesmo vocabulário TF-IDF, avaliação honesta por não ter sido usada
no ajuste do modelo.

**Pré-requisito**: a etapa `labeling` já deve ter sido executada.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from functools import partial

from config.paths import load_project_paths
from config.settings import load_general_config
from constants.labels import ID_TO_LABEL, SENTIMENT_CLASSES
from data.loader import load_training_example_dataset, read_dataset_file
from data.splitter import create_stratified_split
from evaluation.evaluator import evaluate_classifier
from evaluation.reports import (
    build_evaluation_report,
    merge_evaluation_reports,
    save_evaluation_report,
)
from features.lexical import pivot_tfidf_features_to_wide
from models.factory import create_classifier
from pipelines.features import run_features_stage
from pipelines.training_classical import DEFAULT_CLASSICAL_MODEL_NAMES
from training.trainer import Trainer
from visualization.confusion_matrix import plot_confusion_matrix_heatmap
from visualization.roc_pr_curves import plot_roc_curves_one_vs_rest
from visualization.theme import apply_project_theme, save_figure

apply_project_theme()
paths = load_project_paths()
general_config = load_general_config()

## Reconstrução dos artefatos da etapa `features`

Reexecuta `run_features_stage` (idempotente e com semente fixa) para
obter os caminhos canônicos dos artefatos, em vez de supor um nome de
arquivo — evita duplicar a convenção de nomenclatura interna do pipeline.

In [ ]:
feature_artifacts = run_features_stage(
    paths,
    test_size=general_config.data_split.test_size,
    validation_size=general_config.data_split.validation_size,
    random_seed=general_config.data_split.random_state,
)
training_corpus = load_training_example_dataset(feature_artifacts.training_corpus_path)
tfidf_wide = pivot_tfidf_features_to_wide(read_dataset_file(feature_artifacts.tfidf_features_path))
joined = training_corpus.join(tfidf_wide, on="id").sort("id")
feature_columns = [column for column in tfidf_wide.columns if column != "id"]

## Partição interna de holdout

In [ ]:
joined_split = create_stratified_split(
    joined,
    label_column="sentiment_label",
    test_size=0.2,
    validation_size=0.0,
    random_seed=general_config.data_split.random_state,
)
inner_train = joined_split.filter(joined_split["split"] == "treino")
inner_holdout = joined_split.filter(joined_split["split"] != "treino")

X_train = inner_train.select(feature_columns).to_numpy()
y_train = inner_train["sentiment_label"].to_list()
X_holdout = inner_holdout.select(feature_columns).to_numpy()
y_holdout = inner_holdout["sentiment_label"].to_list()
print(f"Treino interno: {len(y_train)} amostra(s); holdout: {len(y_holdout)} amostra(s).")

## Treino e avaliação de cada classificador clássico

In [ ]:
evaluation_reports = []
for model_name in DEFAULT_CLASSICAL_MODEL_NAMES:
    trainer = Trainer(partial(create_classifier, model_name))
    training_result = trainer.fit(X_train, y_train)
    model = training_result.model

    y_pred = model.predict(X_holdout)
    y_score = model.predict_proba(X_holdout)
    if model_name == "gradient_boosting":
        # XGBoost exige rótulos inteiros (ver `pipelines.training_classical`);
        # as colunas de `y_score` já seguem a ordem de `SENTIMENT_CLASSES`
        # (0/1/2), então só as predições pontuais precisam ser convertidas.
        y_pred = [ID_TO_LABEL[label_id] for label_id in y_pred]

    evaluation_result = evaluate_classifier(y_holdout, y_pred, y_score=y_score)
    print(f"{model_name}: {evaluation_result.point_metrics}")

    confusion_figure = plot_confusion_matrix_heatmap(
        evaluation_result.confusion_matrix, title=f"Matriz de Confusão — {model_name}"
    )
    save_figure(
        confusion_figure, f"matriz_confusao_{model_name}", directory=paths.reports_figures_dir
    )

    roc_figure = plot_roc_curves_one_vs_rest(y_holdout, y_score, title=f"Curvas ROC — {model_name}")
    save_figure(roc_figure, f"curvas_roc_{model_name}", directory=paths.reports_figures_dir)

    evaluation_reports.append(build_evaluation_report(model_name, evaluation_result))

classical_report = merge_evaluation_reports(evaluation_reports)
save_evaluation_report(classical_report, paths.reports_metrics_dir / "avaliacao_ml_classico.csv")
classical_report

## Conclusões

Registrar aqui: (1) qual baseline clássico obteve o melhor F1-macro e se a
diferença para os demais está dentro do IC 95% (diferenças sobrepostas
não são conclusivas — usar o teste de McNemar em
`notebooks/07_avaliacao_comparativa.ipynb` para confirmar significância);
(2) padrões de confusão sistemática entre classes adjacentes (ex.:
neutro/positivo); (3) se a diferença de desempenho entre modelos lineares
(regressão logística/SVM linear) e não lineares (random forest/gradient
boosting) justifica a complexidade adicional destes últimos.